## FABlib API References Examples

- [fablib.show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config)
- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [slice.list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodesß)
- [slice.show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete)

## Main purpose

Retrieve pod logs.

In [1]:
import datetime
import json
import asyncio
from configuration import SLICE_NAME
from utils import upload_and_execute_file, override_configuration_files, upload_file, execute_file

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
Bastion Host,bastion.fabric-testbed.net
Bastion Username,apipilikas_0000444352
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


## Setting up variables

In [2]:
%%time
image = "default_ubuntu_24"

# Please adhere to the following regex for naming: /[a-z][a-z0-9]+/
# note: see above, renamed the agent names to only have hyphens, not underscores 

node_configurations = [
    {
        "type": "control",
        "cores": 2,
        "ram": 8,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "dynamos",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "server",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientone",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "LOSA",
        "host": "losa-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clienttwo",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientthree",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w3.fabric-testbed.net",
    },
    {
        "type": "thirdparty",
        "name": "surf",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "TOKY",
        "host": "toky-w3.fabric-testbed.net",
    }
]

sites = list(set([configuration["site"] for configuration in node_configurations]))
agents = [configuration["name"] for configuration in node_configurations if configuration["type"] == "agent"]
thirdparties = [configuration["name"] for configuration in node_configurations if configuration["type"] == "thirdparty"]

def create_node(slice, configuration):
    if (configuration["type"] == "control"): 
        configuration["name"] = "control"

    if (configuration["type"] == "dynamos"): 
        configuration["name"] = "dynamos"
    
    return slice.add_node(name=configuration["name"], 
                          site=configuration["site"], 
                          host=configuration["host"], 
                          cores=configuration["cores"], 
                          ram=configuration["ram"], 
                          disk=configuration["disk"], 
                          validate=True, 
                          raise_exception=True, 
                          image=image)
    

CPU times: user 24 μs, sys: 4 μs, total: 28 μs
Wall time: 36 μs


In [3]:
%%time
slice = fablib.get_slice(name=SLICE_NAME);
nodes = slice.get_nodes();

User: apipilikas@gmail.com bastion key is valid!
Configuration is valid
CPU times: user 177 ms, sys: 27.3 ms, total: 205 ms
Wall time: 4.57 s


In [4]:
# Print ssh information
try:
    # Get slice nodes
    for node in slice.get_nodes():
        print("---------------------------------------------------------------------------")
        print(f"> Node: {node.get_name()}")
        # Get the original SSH command
        original_ssh_command = node.get_ssh_command()
        # Print SSH commands to get into the nodes
        print(f"-- SSH Command from FABRIC:\n   {original_ssh_command}")
        # Replace the file paths in the SSH command
        updated_ssh_command = original_ssh_command.replace(
            "/home/fabric/work/fabric_config/slice_key", "C:/Users/apipi/.ssh/slice_key"
        ).replace(
            "/home/fabric/work/fabric_config/ssh_config", "C:/Users/apipi/.ssh/fabric_ssh_config"
        )
        # Print the updated SSH command
        print(f"-- SSH Command locally:\n   {updated_ssh_command}")

        # Print SSH command forwarding
        token = "%FORWARD%"
        
        ssh_command_parts = updated_ssh_command.split()
        ssh_command_parts.insert(-1, "-L")
        ssh_command_parts.insert(-1, token)

        forward_ssh_command = " ".join(ssh_command_parts)
        api_forward_port = "8080:localhost:8080"
        api_gateway_ssh_command = forward_ssh_command.replace(token, api_forward_port)
        print(f"-- SSH Command forward port api-gateway:\n   {api_gateway_ssh_command}")

        grafana_forward_port = "3000:localhost:3000"
        grafana_ssh_command = forward_ssh_command.replace(token, grafana_forward_port)
        print(f"-- SSH Command forward port grafana:\n   {grafana_ssh_command}")

        prometheus_forward_port = "9090:localhost:9090"
        prometheus_ssh_command = forward_ssh_command.replace(token, prometheus_forward_port)
        print(f"-- SSH Command forward port prometheus:\n   {prometheus_ssh_command}")

        jaeger_forward_port = "16686:localhost:16686"
        prometheus_ssh_command = forward_ssh_command.replace(token, jaeger_forward_port)
        print(f"-- SSH Command forward port jaeger:\n   {prometheus_ssh_command}")

        all_forward_port = f"{api_forward_port} -L {grafana_forward_port} -L {prometheus_forward_port} -L {jaeger_forward_port}"
        all_ssh_command = forward_ssh_command.replace(token, all_forward_port)
        print(f"-- SSH Command forward port ALL:\n   {all_ssh_command}")
    
except Exception as e:
    print(f"Fail: {e}")
    traceback.print_exc()

---------------------------------------------------------------------------
> Node: control
-- SSH Command from FABRIC:
   ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command locally:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port api-gateway:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 8080:localhost:8080 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port grafana:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 3000:localhost:3000 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward port prometheus:
   ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config -L 9090:localhost:9090 ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
-- SSH Command forward

In [5]:
%%time
def get_ip(node):
    interface = node.get_interface(network_name=f"Network-{node.get_site()}")
    return interface.get_ip_addr()

nodes_dict= dict()

for node in nodes[:]:
    ip = get_ip(node)
    name = node.get_name()
    nodes_dict[name] = {"ip": ip, "node": node}
    print(f"{name}: {ip}")

print(nodes_dict)


control: 10.145.1.2
dynamos: 10.145.1.3
server: 10.145.1.4
aggregator: 10.145.1.5
authority: 10.145.1.6
clientone: 10.145.1.7
clienttwo: 10.145.1.8
clientthree: 10.145.1.9
surf: 10.145.1.10
{'control': {'ip': '10.145.1.2', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb3a0d70>}, 'dynamos': {'ip': '10.145.1.3', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38e490>}, 'server': {'ip': '10.145.1.4', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38e0d0>}, 'aggregator': {'ip': '10.145.1.5', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38e210>}, 'authority': {'ip': '10.145.1.6', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38e350>}, 'clientone': {'ip': '10.145.1.7', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38df90>}, 'clienttwo': {'ip': '10.145.1.8', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x7b7cfb38e5d0>}, 'clientthree': {'ip': '1

In [6]:
main_node=nodes_dict['control']['node']
print(f"The main node for now on is [{main_node.get_name()}]")

The main node for now on is [control]


## Initializing retrieving function

Parameters:
- namespace
- search_pod
- container

In [7]:
def retrieve_and_download_logs(namespace, search_pod, container):
    file_name = f"{namespace}-{search_pod}-{container}-logs.txt"
    main_node.execute(f"/home/ubuntu/scattered-directive-energy-monitoring/scripts/retrieve_logs.sh {namespace} {search_pod} {container}")
    main_node.download_file(f"output/logs/{file_name}", f"scattered-directive-energy-monitoring/scripts/logs/{file_name}")

In [8]:
retrieve_and_download_logs("api-gateway", "api-gateway", "api-gateway")


=============== Started retrieving logs ===============


Namespace: api-gateway
Pod start search: api-gateway
Container: api-gateway

Pod name: api-gateway-7494dfc5f6-nrzg9


Saving file api-gateway-api-gateway-api-gateway-logs.txt ...


=============== Finished retrieving logs ===============



In [13]:
retrieve_and_download_logs("clientone", "evangelos-pipilikas", "vfl-party")
retrieve_and_download_logs("server", "evangelos-pipilikas", "vfl-party")
retrieve_and_download_logs("authority", "evangelos-pipilikas", "vfl-authority")
retrieve_and_download_logs("aggregator", "evangelos-pipilikas", "vfl-aggregator")


=============== Started retrieving logs ===============


Namespace: clientone
Pod start search: evangelos-pipilikas
Container: vfl-party

Pod name: evangelos-pipilikas-00aacca7clientone1-kvqs4


Saving file clientone-evangelos-pipilikas-vfl-party-logs.txt ...


=============== Finished retrieving logs ===============


=============== Started retrieving logs ===============


Namespace: server
Pod start search: evangelos-pipilikas
Container: vfl-party

Pod name: evangelos-pipilikas-00aacca7server1-qbw8g


Saving file server-evangelos-pipilikas-vfl-party-logs.txt ...


=============== Finished retrieving logs ===============


=============== Started retrieving logs ===============


Namespace: authority
Pod start search: evangelos-pipilikas
Container: vfl-authority

Pod name: evangelos-pipilikas-00aacca7authority1-97kwr


Saving file authority-evangelos-pipilikas-vfl-authority-logs.txt ...


=============== Finished retrieving logs ===============


=============== Started retrieving

In [10]:
retrieve_and_download_logs("clientone", "evangelos-pipilikas", "vfl-train")
retrieve_and_download_logs("server", "evangelos-pipilikas", "vfl-train-model")


=============== Started retrieving logs ===============


Namespace: clientone
Pod start search: evangelos-pipilikas
Container: vfl-train

Pod name: evangelos-pipilikas-85c0cfdfclientone1-2hmqv


Saving file clientone-evangelos-pipilikas-vfl-train-logs.txt ...


=============== Finished retrieving logs ===============


=============== Started retrieving logs ===============


Namespace: server
Pod start search: evangelos-pipilikas
Container: vfl-train-model

Pod name: evangelos-pipilikas-85c0cfdfserver1-tj59g


Saving file server-evangelos-pipilikas-vfl-train-model-logs.txt ...


=============== Finished retrieving logs ===============

